In [2]:
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import requests
import time
import os
!pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


True

In [3]:
onemap_api_key = os.environ.get('ONEMAP_API_KEY')

if not onemap_api_key:
    raise ValueError(
        "ONEMAP_API_KEY not found. Please create a .env file in this directory with: \n"
        "ONEMAP_API_KEY=your_api_key_here"
    )

In [4]:
# Get file names 
hdb_points_file = "resale_2025_centroids.geojson"
busstop_file = "LTABusStop.geojson"
output_file = "hdb_busstop_distances.csv"

### Run a sample of postal codes to ensure code works, check the result output to see if there are any nulls 

In [5]:
# --- 1. load and prep data ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
# take only the first 5 unique postals for this sample
postal_gdf_sample = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).head(5).copy()

busstop_gdf = gpd.read_file(busstop_file).to_crs(epsg=4326)

busstop_coords = list(zip(busstop_gdf.geometry.y, busstop_gdf.geometry.x))
busstop_nums = busstop_gdf['BUS_STOP_NUM'].tolist() 

tree = cKDTree(busstop_coords)

results = []

print(f"--- starting sample test for {len(postal_gdf_sample)} postals ---")

# --- 2. sample loop ---
for idx, row in postal_gdf_sample.iterrows():
    p_code = str(row['postal_code'])
    origin_coords = (row.geometry.y, row.geometry.x)
    
    print(f"\nchecking postal: {p_code} at {origin_coords}")
    
    # get 3 closest bus stop by straight line
    dist, indices = tree.query(origin_coords, k=3) 
    
    best_walk = float('inf')
    closest_busstop = None

    for i in indices:
        dest_coords = busstop_coords[i]
        busstop_num = busstop_nums[i]
        
        
        # using the routingsvc endpoint from the docs
        url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
        
        try:

            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            headers = {"Authorization": token}
        
            res = requests.get(url, headers=headers, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                if 'route_summary' in data:
                    walk_m = data['route_summary']['total_distance']
                    print(f"  -> found route to {busstop_num}: {walk_m}m")
                    if walk_m < best_walk:
                        best_walk = walk_m
                        closest_busstop = busstop_num
                else:
                    print(f"  -> no route summary found for {busstop_num}")
            else:
                print(f"  -> api error {res.status_code}: {res.text}")
            
            time.sleep(0.2) # slightly longer sleep for the sample
        except Exception as e:
            print(f"  -> request failed: {e}")

    results.append({
        "postal_code": p_code,
        "nearest_busstop": closest_busstop,
        "walking_dist_m": best_walk if best_walk != float('inf') else None
    })

# --- 3. display output ---
sample_df = pd.DataFrame(results)
print("\n--- final sample output ---")
print(sample_df)

# check if we actually got results
if sample_df['walking_dist_m'].isnull().all():
    print("\nwarning: all walking distances are null. check your api key and coordinate order.")
else:
    print("\nsuccess: distances found! you can now run the full script.")

--- starting sample test for 5 postals ---

checking postal: 560406 at (1.362004539458907, 103.8538799102005)
  -> found route to 54319: 158m
  -> found route to 54311: 230m
  -> found route to 54371: 303m

checking postal: 560314 at (1.366227071213243, 103.85008585919796)
  -> found route to 54331: 292m
  -> found route to 54339: 522m
  -> found route to 54329: 599m

checking postal: 561308 at (1.365483427541735, 103.84408473990418)
  -> found route to 54011: 317m
  -> found route to 54019: 130m
  -> found route to 54071: 203m

checking postal: 560323 at (1.36790849370682, 103.84771408808119)
  -> found route to 54247: 154m
  -> found route to 54248: 220m
  -> found route to 54241: 276m

checking postal: 560174 at (1.375196558818988, 103.83769584780252)
  -> found route to 54199: 445m
  -> found route to 54191: 257m
  -> found route to 54201: 415m

--- final sample output ---
  postal_code nearest_busstop  walking_dist_m
0      560406           54319             158
1      560314     

### Run full script if results generated for the sample postal codes

In [ ]:
# --- 1. Load Datasets (full) ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
postal_gdf = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).copy()

busstop_gdf = gpd.read_file(busstop_file).to_crs(epsg=4326)
busstop_coords = list(zip(busstop_gdf.geometry.y, busstop_gdf.geometry.x))
busstop_nums = busstop_gdf['BUS_STOP_NUM'].tolist()

tree = cKDTree(busstop_coords)

# --- 2. Resume logic ---
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    processed_postals = set(processed_df['postal_code'].astype(str))
    results = processed_df.to_dict('records')
    print(f"Resuming... {len(processed_postals)} postals already completed.")
else:
    results = []
    processed_postals = set()

# --- 3. Full loop ---
print(f"Starting processing for {len(postal_gdf) - len(processed_postals)} remaining postals...")

try:
    for idx, row in postal_gdf.iterrows():
        p_code = row['postal_code']
        if p_code in processed_postals:
            continue

        origin_coords = (row.geometry.y, row.geometry.x)
        dist, indices = tree.query(origin_coords, k=3)
        
        best_walk = float('inf')
        best_busstop = None

        for i in indices:
            dest_coords = busstop_coords[i]
            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
            
            try:
                res = requests.get(url, headers={"Authorization": token}, timeout=10)
                if res.status_code == 200:
                    data = res.json()
                    if 'route_summary' in data:
                        walk_m = data['route_summary']['total_distance']
                        if walk_m < best_walk:
                            best_walk = walk_m
                            best_busstop = busstop_nums[i]
                time.sleep(0.2)
            except:
                continue

        results.append({
            "postal_code": p_code,
            "nearest_busstop": best_busstop,
            "walking_dist_m": best_walk if best_walk != float('inf') else None
        })
        processed_postals.add(p_code)

        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"progress: {len(results)} / {len(postal_gdf)} unique postals saved.")
            

except KeyboardInterrupt:
    print("\nPaused. Saving current progress...")

# Final save
pd.DataFrame(results).to_csv(output_file, index=False)
print(f"Task complete! Results saved to {output_file}")

Starting processing for 8023 remaining postals...
progress: 50 / 8023 unique postals saved.
progress: 100 / 8023 unique postals saved.
progress: 150 / 8023 unique postals saved.
progress: 200 / 8023 unique postals saved.
progress: 250 / 8023 unique postals saved.
progress: 300 / 8023 unique postals saved.
progress: 350 / 8023 unique postals saved.
progress: 400 / 8023 unique postals saved.
progress: 450 / 8023 unique postals saved.
progress: 500 / 8023 unique postals saved.
progress: 550 / 8023 unique postals saved.
progress: 600 / 8023 unique postals saved.
progress: 650 / 8023 unique postals saved.
progress: 700 / 8023 unique postals saved.
progress: 750 / 8023 unique postals saved.
progress: 800 / 8023 unique postals saved.
progress: 850 / 8023 unique postals saved.
progress: 900 / 8023 unique postals saved.
progress: 950 / 8023 unique postals saved.
progress: 1000 / 8023 unique postals saved.
progress: 1050 / 8023 unique postals saved.
progress: 1100 / 8023 unique postals saved.
pr

In [ ]:
full_hdb_point = gpd.read_file("resale_2025_centroids.geojson")
hdb_busstop_distances = pd.read_csv("hdb_busstop_distances.csv")

In [ ]:
hdb_busstop_distances.info()

In [ ]:
# 1. Force the postal_code in the GeoDataFrame to be a string
full_hdb_point['postal_code'] = full_hdb_point['postal_code'].astype(str).str.zfill(6)

# 2. Force the postal_code in the distance CSV to be a string
hdb_busstop_distances['postal_code'] = hdb_busstop_distances['postal_code'].astype(str).str.zfill(6)

# 3. Now run the merge
hdb_resale_with_busstop = full_hdb_point.merge(
    hdb_busstop_distances[['postal_code', 'nearest_busstop', 'walking_dist_m']], 
    on='postal_code', 
    how='left'
)

print("Merge complete!")

In [ ]:
# Check for null entries
hdb_resale_with_busstop.info()
hdb_resale_with_busstop.isnull().any()

### Checking for any anomalies, then verify on onemap if walking distance is too big

In [ ]:
hdb_resale_with_busstop[hdb_resale_with_busstop["walking_dist_m"] > 2500]

### Save results to file

In [ ]:
# hdb_resale_with_busstop.to_file("hdb_resale_busstop.geojson", driver="GeoJSON")

In [ ]:
hdb_resale_with_busstop.to_csv("hdb_resale_busstop.csv")

In [ ]:
len(hdb_resale_with_busstop)